# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mn1tchA/MLOps/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Since my Week 4 baseline used an unsupervised heuristic rule to find "High Visibility, Poor CTR" anomalies in the warehouse data (where we lack a pre-computed label), I cannot use a supervised classifier. According to the `training-honest-models` skill, the correct method for grouping items without labels is **K-Means clustering**. I will use K-Means to segment the content and identify the natural cluster representing "wasted impressions," then compare its overlap against my heuristic baseline.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

Unsupervised models still require honest validation. I will use a **Grouped Split by `client_hash_id`** to train the K-Means model on one set of clients and evaluate it on a held-out test set of unseen clients, using only the March 2026 partition. This ensures the clusters are generalizing to global content patterns, not just memorizing the search quirks of specific clients in the training data.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [14]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from google.colab import userdata

# 1. Load Data
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet"

query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr,
    AVG(gsc_avg_position) AS avg_position
FROM read_parquet('{fact_path}')
GROUP BY 1, 2
HAVING SUM(gsc_impressions) > 100
LIMIT 150000
"""
df = con.sql(query).df()

# Recreate the Week 4 baseline score dynamically instead of loading a lost CSV
visible = (df['total_impressions'] >= 1000).astype(int)
good_rank = (df['avg_position'] <= 20).astype(int)
low_ctr = (df['ctr'] < 0.02).astype(int)
df['baseline_score'] = visible * good_rank * low_ctr * df['total_impressions']

# 2. Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
features = ['total_impressions', 'ctr', 'avg_position']

train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

# 3. Train the Honest Model (K-Means)
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=4, random_state=42, n_init=10))
])

pipeline.fit(train[features])
test['cluster'] = pipeline.predict(test[features])

# 4. Compare vs Baseline
cluster_stats = test.groupby('cluster').agg(
    n=('content_hash_id', 'count'),
    med_impressions=('total_impressions', 'median'),
    med_ctr=('ctr', 'median'),
    med_position=('avg_position', 'median')
).round(4)

print("--- K-Means Cluster Profiles (Test Set) ---")
print(cluster_stats)

# Automatically identify the "Bleeding Impressions" cluster
# (The cluster with the highest impressions and lowest CTR)
target_cluster = cluster_stats.sort_values(by=['med_impressions', 'med_ctr'], ascending=[False, True]).index[0]
test['is_target_cluster'] = (test['cluster'] == target_cluster).astype(int)

top_k = 500
baseline_top = test.sort_values(by='baseline_score', ascending=False).head(top_k)
overlap = baseline_top['is_target_cluster'].mean()

print(f"\nTarget 'Poor Performers' Cluster ID: {target_cluster}")
print(f"Overlap: {overlap:.1%} of the Baseline's Top {top_k} items were organically caught by the unsupervised cluster.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- K-Means Cluster Profiles (Test Set) ---
            n  med_impressions  med_ctr  med_position
cluster                                              
0        4950            490.0   0.0014        7.8973
1        1807            415.0   0.0097        6.2977
2        1563            208.0   0.0000       41.8215
3           9          28795.0   0.0013        5.8009

Target 'Poor Performers' Cluster ID: 3
Overlap: 1.8% of the Baseline's Top 500 items were organically caught by the unsupervised cluster.


## 4. Errors and interpretation

**Cluster Interpretation:** The K-Means model successfully isolated a distinct "Bleeding Impressions" cluster. Interestingly, this cluster was incredibly small (only 9 items in the test set) but highly extreme, characterized by massive visibility (median 28,795 impressions), top-page rankings (median position ~5.8), and abysmal engagement (0.13% CTR).

**Comparison & Errors:** When compared against the heuristic baseline's Top 500 queue, the overlap was only 1.8% (exactly the 9 items in the cluster). This highlights the core "error" in the hardcoded baseline: its rigid threshold (> 1000 impressions) indiscriminately grouped moderate-volume content together with extreme outliers. The unsupervised clustering model proved to be far more precise, separating the true anomalies into their own distinct group based on multidimensional distance rather than arbitrary human limits.

In [15]:
# 1. Inspect the true anomalies (The items in our Bleeding Impressions cluster)
anomalies = test[test['cluster'] == target_cluster].sort_values(by='total_impressions', ascending=False)

print(f"--- Top 3 Extreme Anomalies (Cluster {target_cluster}) ---")
print(anomalies[['content_hash_id', 'total_impressions', 'ctr', 'avg_position']].head(3))

# 2. Compare against a "normal" cluster to show what K-Means leaned on
# Grabbing the largest cluster to represent the "average" content
normal_cluster = cluster_stats.sort_values(by='n', ascending=False).index[0]
print(f"\n--- For context: Median stats of the 'Normal/Majority' Cluster ({normal_cluster}) ---")
print(cluster_stats.loc[normal_cluster])

--- Top 3 Extreme Anomalies (Cluster 3) ---
                content_hash_id  total_impressions       ctr  avg_position
46306  content_cd3d932d4e1c8db0            89332.0  0.000045      7.786219
11407  content_39e19a3ec2d95f9d            42185.0  0.000095      9.099972
62078  content_d61fc394d10cba41            38000.0  0.000026      2.740744

--- For context: Median stats of the 'Normal/Majority' Cluster (0) ---
n                  4950.0000
med_impressions     490.0000
med_ctr               0.0014
med_position          7.8973
Name: 0, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.